## 01c — Spender Behaviour

Part 1(c) of the assignment.

This notebook analyses:
1. Revenue concentration among players
2. Time from install to first purchase
3. Share of players who never purchase
4. Time between consecutive purchases for repeat buyers

In [33]:
import sys
sys.path.append("..")

import pandas as pd
from src.db import get_connection

conn = get_connection()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Database connected successfully.")

Database connected successfully.


In [34]:
import re

def split_sql_blocks(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    
    text = re.sub(r'^\s*--.*$', '', text, flags=re.MULTILINE)

    
    blocks = [
        q.strip()
        for q in text.split(';')
        if q.strip()
    ]

    return blocks


blocks_1c = split_sql_blocks('../sql/1c_spender_behaviour.sql')

print(f"Found {len(blocks_1c)} query blocks")

assert len(blocks_1c) == 4

Found 4 query blocks


## 1. Revenue Concentration

Revenue concentration is calculated among paying players only.

I use payers as the ranking population because the purpose of this metric is
to understand how concentrated realized IAP revenue is among players who
actually monetize. Players with zero revenue are excluded from the ranking,
while total revenue from the payer population is used as the denominator.

For transparency, the SQL calculates concentration for both payers-only and
all-player populations, but the payers-only result is reported as the primary
metric.

In [35]:
print(" Revenue concentration ")

df_1c_1 = pd.read_sql_query(blocks_1c[0], conn)


df_1c_1 = df_1c_1[
    df_1c_1["population"] == "payers_only"
].copy()

display(df_1c_1)

 Revenue concentration 


,population,share_top_1pct,share_top_10pct,share_top_50pct
0,payers_only,0.0944,0.4808,0.9023


In [36]:
print("Time to first purchase ")

df_1c_2 = pd.read_sql_query(blocks_1c[1], conn)

display(df_1c_2)

Time to first purchase 


,bucket,n_players,pct_of_payers
0,0 (day of install),638,5.8
1,1 day,471,4.3
2,2-3 days,727,6.6
3,4-7 days,1211,11.1
4,8-14 days,1337,12.2
5,15-30 days,1842,16.8
6,31-60 days,1669,15.3
7,60+ days,3049,27.9


### Data-quality handling

110 payers have a recorded first purchase 1–4 days before their recorded install date. These negative time-to-first-purchase observations are treated as an install-date / attribution data-quality issue and excluded from the timing distribution rather than interpreted as valid player behaviour.

The time-to-first-purchase distribution therefore uses 10,944 valid payers, while the overall ever-paid / never-paid calculation retains all 11,054 observed payers.

In [37]:
print(" Never-pay share ")

df_1c_3 = pd.read_sql_query(blocks_1c[2], conn)

display(df_1c_3)

 Never-pay share 


,total_players,ever_paid,pct_never_pay
0,80295,11054,0.8623


In [38]:
print(" Repeat purchase gap ")

df_1c_4 = pd.read_sql_query(blocks_1c[3], conn)

display(df_1c_4)

 Repeat purchase gap 


,gap_bucket,n_gaps,pct_of_gaps,avg_days_in_bucket
0,same day,393,9.0,0.4
1,1-3 days,402,9.2,1.9
2,30+ days,1588,36.3,39.1
3,4-7 days,450,10.3,5.6
4,8-14 days,608,13.9,10.7
5,15-30 days,931,21.3,21.5


In [39]:
share_cols = [
    "share_top_1pct",
    "share_top_10pct",
    "share_top_50pct"
]

for col in share_cols:
    assert df_1c_1[col].between(0, 1).all(), \
        f"ERROR: {col} outside [0, 1]"

print("Revenue concentration check passed.")

Revenue concentration check passed.


In [40]:
df_1c_1.to_csv(
    '../outputs/tables/01c_revenue_concentration.csv',
    index=False
)

df_1c_2.to_csv(
    '../outputs/tables/01c_time_to_first_purchase.csv',
    index=False
)

df_1c_3.to_csv(
    '../outputs/tables/01c_never_pay_share.csv',
    index=False
)

df_1c_4.to_csv(
    '../outputs/tables/01c_repeat_purchase_gap.csv',
    index=False
)

print("All 01c outputs saved successfully.")

All 01c outputs saved successfully.
